数据预处理

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('merged_result_with_depression_mentaldisease.csv')
merged = pd.read_csv(r"mental_disease_with_followup.csv")
protein = pd.merge(df,merged,on='Participant ID', how='inner')  
protein

In [ ]:
participant_ids = protein['Participant ID']
base = pd.read_csv('1Baseline_characteristics_participant.csv')
base = base[base['Participant ID'].isin(participant_ids)]
base[['Participant ID','Age at recruitment','Sex','Townsend deprivation index at recruitment']]
Sociodemographics = pd.read_csv('3Sociodemographics.csv')
Sociodemographics = Sociodemographics[Sociodemographics['Participant ID'].isin(participant_ids)]
Sociodemographics[['Participant ID','Ethnic background | Instance 0']]
Physical = pd.read_csv('6-1Body_size_measures_participant.csv')
Physical = Physical[Physical['Participant ID'].isin(participant_ids)]
Physical[['Participant ID','Body mass index (BMI) | Instance 0']]
Blood = pd.read_csv('Biological_sampling_participant.csv')
Blood = Blood[Blood['Participant ID'].isin(participant_ids)]
Blood[['Participant ID','Time blood sample collected | Instance 0 | Array 0','Fasting time | Instance 0']]

In [ ]:
# First merge protein with Blood
protein_merged = pd.merge(protein, 
                         Blood[['Participant ID','Time blood sample collected | Instance 0 | Array 0','Fasting time | Instance 0']], 
                         on='Participant ID', 
                         how='left')

# Then merge with Physical
protein_merged = pd.merge(protein_merged, 
                         Physical[['Participant ID','Body mass index (BMI) | Instance 0']], 
                         on='Participant ID', 
                         how='left')

# Then merge with Sociodemographics
protein_merged = pd.merge(protein_merged, 
                         Sociodemographics[['Participant ID','Ethnic background | Instance 0']], 
                         on='Participant ID', 
                         how='left')

# Finally merge with base
protein_merged = pd.merge(protein_merged, 
                         base[['Participant ID','Age at recruitment','Sex','Townsend deprivation index at recruitment']], 
                         on='Participant ID', 
                         how='left')

# Drop unnecessary columns related to diagnosis details 
# that are already summarized in the depression status flags
columns_to_drop = ['before_x', 'after_x', 'combined_values', 'before_y', 'after_y', 'depressed']
protein_merged = protein_merged.drop(columns=columns_to_drop, errors='ignore')

# Display the cleaned DataFrame
# Convert the time column to datetime and extract season
protein_merged['Blood_Collection_Time'] = pd.to_datetime(protein_merged['Time blood sample collected | Instance 0 | Array 0'])
protein_merged['Blood_Collection_Month'] = protein_merged['Blood_Collection_Time'].dt.month

# Map months to seasons: Spring(0), Summer(1), Fall(2), Winter(3)
season_mapping = {
    1: 3, 2: 3,  # Winter: Jan, Feb
    3: 0, 4: 0, 5: 0,  # Spring: Mar, Apr, May
    6: 1, 7: 1, 8: 1,  # Summer: Jun, Jul, Aug
    9: 2, 10: 2, 11: 2,  # Fall: Sep, Oct, Nov
    12: 3  # Winter: Dec
}

protein_merged['Blood_Collection_Season'] = protein_merged['Blood_Collection_Month'].map(season_mapping)
protein_merged

In [ ]:
mental = pd.read_csv('disease.csv')
protein_merged = pd.merge(protein_merged, mental[['Participant ID','menta_date_after' , 'depressed_date_after','dementia_date_after','bd_date_after','anxiety_date_after','scz_date_after','sleep_date_after','sud_date_after','sd_date_after',
                                                  'menta_before','menta_after','depressed-after','dementia','bd','anxiety','scz','sleep','sud','sd']], on='Participant ID', how='inner')  
protein_merged.drop(columns=['followup_years','icd','Years Difference'], inplace=True)

In [ ]:
sup = pd.read_csv("supplement3_instance_0.csv")
protein_merged = pd.merge(sup, protein_merged, on='Participant ID', how='inner')
protein_merged

In [ ]:
protein_merged.to_csv('protein_merged_mental.csv', index=False)

## cox后插补

In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv('protein_merged_mental.csv',index_col='Participant ID')
data['any_before'] = np.where(
    (data['menta_before'] == 1) | (data['mental_before'] == 1) , 1,0)
data['any_after'] = np.where(
    (data['menta_after'] == 1) | (data['mental'] == 1) , 1,0)
data = data[(data['any_before'] != 1)]
data

In [ ]:
# Create a dictionary for ethnic background mapping
ethnic_mapping = {
    1001: 1, 2001: 1, 3001: 1, 4001: 1,
    1002: 2, 2002: 2, 3002: 2, 4002: 2,
    1003: 3, 2003: 3, 3003: 3, 4003: 3, 5: 3,
    2004: 4, 3004: 4,
    6: 0, -1: 0, -3: 0
}

# Apply the mapping to the column
data['Ethnic background | Instance 0'] = data['Ethnic background | Instance 0'].map(ethnic_mapping)

In [ ]:
# 选择需要处理的列
cols_to_process = data.columns.tolist()
data_subset = data[cols_to_process]
data_subset.dropna(thresh=data.shape[0]*0.7, axis=1, inplace=True)

other_cols = [col for col in data.columns if col not in cols_to_process]
data = pd.concat([data_subset, data[other_cols]], axis=1)
data.shape

In [ ]:
data_subset.columns.tolist()[2916:2924]

In [ ]:
# # 选择需要处理的列
# cols_to_process = data_subset.columns.tolist()
# data_subset = data[cols_to_process]
# data_subset.dropna(thresh=data.shape[1]*0.49, inplace=True)
# other_cols = [col for col in data.columns if col not in cols_to_process]
# data_enc = pd.merge(data_subset, data[other_cols],on='Participant ID', how='left')
# data_enc.shape

In [ ]:
data.columns.tolist()[2910:2921]

In [ ]:
myout_df= pd.read_csv('result/Dep_Cox_improved_any.csv')
# Filter proteins with significant FDR-corrected p-values
significant_proteins = myout_df[myout_df['p_val_fdr'] < 0.01]
#significant_proteins = significant_proteins[(significant_proteins['HR'] < 0.5) | (significant_proteins['HR'] > 1.095)]
# Sort by p-value (to show most significant first)
significant_proteins = significant_proteins.sort_values(by='p_val_fdr')

print(f"Found {len(significant_proteins)} significant proteins (FDR < 0.05)")
# Extract list of significant proteins
data1 = data
data1.rename(columns={'depressed-after':'depressed','any_after':'any'}, inplace=True)
sig_proteins_list = [p for p in (significant_proteins['Pro_code'].tolist()) if p in (data1.columns.tolist())]

# Create a dataframe with only significant proteins, Participant ID, and depressed-after
selected_columns = ['any', 'depressed', 'dementia', 'bd', 'anxiety', 'scz', 'sleep', 'sud', 'sd','Region_Code','LEP'] + sig_proteins_list
significant_data = data1[selected_columns]
print(f"Created dataset with {len(sig_proteins_list)} significant proteins")
print(f"Dataset shape: {significant_data.shape}")

# Display first few rows
significant_data.head()

In [ ]:
data_enc = significant_data.copy()

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
pro_f_lst = data_enc.columns.tolist()[10:]
scaler = StandardScaler()
scaler.fit(data_enc[pro_f_lst])
tmp = scaler.transform(data_enc[pro_f_lst])
df_prot_train_tissue = pd.DataFrame(tmp, index=data_enc[pro_f_lst].index, columns=data_enc[pro_f_lst].columns)
other_cols = [col for col in data_enc.columns if col not in pro_f_lst]
data_enc = pd.concat([df_prot_train_tissue, data_enc[other_cols]], axis=1)            
joblib.dump(scaler, 'result/model/protein_zscore_scaler.pkl',compress=3)

In [ ]:
# 对于数值变量使用中位数，分类变量使用众数
from sklearn.impute import SimpleImputer

# 设定一个阈值，唯一值数量少于这个值的数字列被视为分类变量
max_unique_for_categorical = 20

numeric_cols = []
categorical_cols = []

for col in data1.columns:
    if data1[col].dtype in [np.int64, np.float64]:
        # 数字列：检查唯一值数量
        if data1[col].nunique() <= max_unique_for_categorical:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)
    else:
        # 非数字列直接视为分类变量
        categorical_cols.append(col)

print("数值列:", numeric_cols)
print("分类列:", categorical_cols)

# 然后进行插补（同上）

# 分别插补
if len(numeric_cols) > 0:
    num_imputer = SimpleImputer(strategy='most_frequent')
    data1[numeric_cols] = num_imputer.fit_transform(data1[numeric_cols])
    
if len(categorical_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    data1[categorical_cols] = cat_imputer.fit_transform(data1[categorical_cols])
data1

In [ ]:
data1.to_csv('significant_proteins_data_processed1.csv', index=True)